In [ ]:
import os
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

DATASET_PATH = "../dataset/ravdess"

print(os.listdir(DATASET_PATH))

In [ ]:
emotion_labels = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

data = []

for actor in os.listdir(DATASET_PATH):
    actor_path = os.path.join(DATASET_PATH, actor)

    for filename in os.listdir(actor_path):
        if filename.endswith(".wav"):
            emotion_code = filename.split("-")[2]

            emotion = emotion_labels[emotion_code]

            file_path = os.path.join(actor_path, filename)

            data.append({
                "file_path": file_path,
                "filename": filename,
                "emotion": emotion
            })

df = pd.DataFrame(data)

print(df.head())
print("Total audio files:", len(df))

In [ ]:
audio_path = df.iloc[0]["file_path"]

audio, sample_rate = librosa.load(
    audio_path,
    sr=22050
)

mfcc = librosa.feature.mfcc(
    y=audio,
    sr=sample_rate,
    n_mfcc=40
)

print("MFCC shape:", mfcc.shape)

In [ ]:
plt.figure(figsize=(12, 5))

librosa.display.specshow(
    mfcc,
    sr=sample_rate,
    x_axis="time"
)

plt.colorbar(format="%+2.0f dB")

plt.title("MFCC Visualization")
plt.xlabel("Time")
plt.ylabel("MFCC Coefficients")

plt.show()

In [ ]:
mfcc_mean = np.mean(mfcc, axis=1)

print("MFCC feature vector:")
print(mfcc_mean)

print("Feature vector shape:", mfcc_mean.shape)

In [ ]:
frame_lengths = []

for file_path in df["file_path"]:

    audio, sr = librosa.load(
        file_path,
        sr=22050
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40
    )

    frame_lengths.append(mfcc.shape[1])

print("Minimum frames:", min(frame_lengths))
print("Maximum frames:", max(frame_lengths))

In [ ]:
X = []
y = []

for file_path, emotion in zip(
    df["file_path"],
    df["emotion"]
):

    audio, sr = librosa.load(
        file_path,
        sr=22050
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40
    )

    # Pad or truncate to fixed frame length (228)
    MAX_FRAMES = 228
    if mfcc.shape[1] < MAX_FRAMES:
        pad_width = MAX_FRAMES - mfcc.shape[1]
        mfcc = np.pad(
            mfcc,
            ((0, 0), (0, pad_width)),
            mode="constant"
        )
    else:
        mfcc = mfcc[:, :MAX_FRAMES]

    X.append(mfcc)
    y.append(emotion)

X = np.array(X)
y = np.array(y)

print("Features shape:", X.shape)
print("Labels shape:", y.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Emotion classes:", label_encoder.classes_)
print("Encoded labels:", y_encoded[:10])

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Extract actor IDs from filenames
groups = df["filename"].str.split("-").str[-1].str.replace(
    ".wav",
    "",
    regex=False
).astype(int)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y_encoded, groups)
)

X_train, X_test = X[train_idx], X[test_idx]

y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

In [ ]:
# Standardize using training data statistics

mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std

X_test = (X_test - mean) / std

# Add channel dimension

X_train = X_train[..., np.newaxis]

X_test = X_test[..., np.newaxis]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

In [ ]:
np.savez(
    "audio_features.npz",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    groups=groups.to_numpy(),
    train_idx=train_idx,
    test_idx=test_idx,
    classes=label_encoder.classes_,
    norm_mean=mean,
    norm_std=std
)

np.savez(
    "../models/audio_features.npz",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    groups=groups.to_numpy(),
    train_idx=train_idx,
    test_idx=test_idx,
    classes=label_encoder.classes_,
    norm_mean=mean,
    norm_std=std
)